[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Testing Failure &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the project's folders and `run_pytest`, and the cell after it writes the
module as the notebook left it. Run them first, then the tasks in order, since tasks 2 to 4 add to
the file task 1 writes. The last cell removes the scratch folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import math
import statistics
import warnings


class ReadingError(ValueError):
    """A line that is not a reading. line_number is the line's number in its file, when it is known."""

    def __init__(self, message, line_number=None):
        super().__init__(message)
        self.line_number = line_number


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    A line that is not a reading raises ReadingError.
    """
    fields = line.strip().split(",")
    if len(fields) != 2:
        raise ReadingError(f"expected 2 fields, got {len(fields)}: {fields}")
    station, celsius = fields
    if not station:
        raise ReadingError(f"no station name: {line.strip()!r}")
    celsius = celsius.replace("\u2212", "-")
    if not celsius:
        return station, None
    try:
        temperature = float(celsius)
    except ValueError:
        raise ReadingError(f"not a temperature: {celsius!r}") from None
    if not math.isfinite(temperature):
        raise ReadingError(f"not a temperature: {celsius!r}")
    return station, temperature


def parse_line(line):
    """The old name of parse_reading, kept for code that still uses it."""
    warnings.warn("parse_line is deprecated; use parse_reading", DeprecationWarning, stacklevel=2)
    return parse_reading(line)


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped.

    A line that is not a reading raises ReadingError, with the line's number.
    """
    by_station = {}
    for number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            station, celsius = parse_reading(line)
        except ReadingError as error:
            raise ReadingError(f"line {number}: {error}", line_number=number) from error
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def summarize_file(path):
    """Each station's mean temperature, from a file of readings."""
    with open(path, encoding="utf-8-sig") as file:
        return summarize(file)


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


**1.** A line with no comma.


In [3]:
%%writefile scratch/stations/tests/test_tasks.py
import pytest

from readings import ReadingError, parse_reading


def test_a_station_alone_raises_reading_error():
    with pytest.raises(ReadingError):
        parse_reading("Oslo")


Writing scratch/stations/tests/test_tasks.py


In [4]:
run_pytest("tests/test_tasks.py", "-q")


.                                                                        [100%]
1 passed


The call is the only line in the block, and the test needs nothing after it.


**2.** The message, with match.


In [5]:
%%writefile scratch/stations/tests/test_tasks.py
import pytest

from readings import ReadingError, parse_reading


def test_a_station_alone_raises_reading_error():
    with pytest.raises(ReadingError):
        parse_reading("Oslo")


def test_a_word_is_not_a_temperature():
    with pytest.raises(ReadingError, match="not a temperature: 'cold'"):
        parse_reading("Oslo,cold")


Overwriting scratch/stations/tests/test_tasks.py


In [6]:
run_pytest("tests/test_tasks.py", "-q")


..                                                                       [100%]
2 passed


The pattern holds nothing with a meaning of its own in a regular expression, so it needs no
`re.escape`.


**3.** ValueError, for a ReadingError.


In [7]:
%%writefile scratch/stations/tests/test_tasks.py
import pytest

from readings import ReadingError, parse_reading


def test_a_station_alone_raises_reading_error():
    with pytest.raises(ReadingError):
        parse_reading("Oslo")


def test_a_word_is_not_a_temperature():
    with pytest.raises(ReadingError, match="not a temperature: 'cold'"):
        parse_reading("Oslo,cold")


def test_an_extra_field_raises_value_error():
    with pytest.raises(ValueError):
        parse_reading("Oslo,-2.4,extra")


Overwriting scratch/stations/tests/test_tasks.py


In [8]:
run_pytest("tests/test_tasks.py", "-q")


...                                                                      [100%]
3 passed


It passes because `ReadingError` is a subclass of `ValueError`, and `pytest.raises` accepts a
subclass of the class it is given. The test says less than it could: it would also pass for a plain
`ValueError`.


**4.** The line number, with excinfo.


In [9]:
%%writefile scratch/stations/tests/test_tasks.py
import pytest

from readings import ReadingError, parse_reading, summarize


def test_a_station_alone_raises_reading_error():
    with pytest.raises(ReadingError):
        parse_reading("Oslo")


def test_a_word_is_not_a_temperature():
    with pytest.raises(ReadingError, match="not a temperature: 'cold'"):
        parse_reading("Oslo,cold")


def test_an_extra_field_raises_value_error():
    with pytest.raises(ValueError):
        parse_reading("Oslo,-2.4,extra")


def test_the_third_line_is_named():
    with pytest.raises(ReadingError) as excinfo:
        summarize(["Oslo,-2.4", "", "Oslo,cold"])

    assert excinfo.value.line_number == 3


Overwriting scratch/stations/tests/test_tasks.py


In [10]:
run_pytest("tests/test_tasks.py", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_tasks.py::test_a_station_alone_raises_reading_error PASSED    [ 25%]
tests/test_tasks.py::test_a_word_is_not_a_temperature PASSED             [ 50%]
tests/test_tasks.py::test_an_extra_field_raises_value_error PASSED       [ 75%]
tests/test_tasks.py::test_the_third_line_is_named PASSED                 [100%]

============================== 4 passed ===============================


The `assert` sits after the block, where it runs.


**5.** Three bad temperatures, parametrized.


In [11]:
%%writefile scratch/stations/tests/test_temperatures.py
import re

import pytest

from readings import ReadingError, parse_reading


@pytest.mark.parametrize("temperature", ["cold", "inf", "-"], ids=["a word", "infinity", "a minus sign alone"])
def test_a_bad_temperature_raises_reading_error(temperature):
    message = f"not a temperature: {temperature!r}"

    with pytest.raises(ReadingError, match=re.escape(message)):
        parse_reading(f"Oslo,{temperature}")


Writing scratch/stations/tests/test_temperatures.py


In [12]:
run_pytest("tests/test_temperatures.py", "-v")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_temperatures.py::test_a_bad_temperature_raises_reading_error[a word] PASSED [ 33%]
tests/test_temperatures.py::test_a_bad_temperature_raises_reading_error[infinity] PASSED [ 66%]
tests/test_temperatures.py::test_a_bad_temperature_raises_reading_error[a minus sign alone] PASSED [100%]

============================== 3 passed ===============================


`{temperature!r}` puts the quotes around the temperature, as the module's message does, and
`re.escape` keeps the pattern right for any temperature added later, although quotes and a lone
minus sign mean nothing special to a regular expression.


**6.** The warning and the value.


In [13]:
%%writefile scratch/stations/tests/test_old_name_task.py
import pytest

from readings import parse_line


def test_the_old_name_warns_and_parses():
    with pytest.warns(DeprecationWarning):
        result = parse_line("Oslo,-2.4")

    assert result == ("Oslo", -2.4)


Writing scratch/stations/tests/test_old_name_task.py


In [14]:
run_pytest("tests/test_old_name_task.py", "-q")


.                                                                        [100%]
1 passed


The call stays inside the block, so that the warning is expected, and the value it returned is
checked after the block.

Last, remove the scratch folder:


In [15]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Testing Failure](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/06-testing-failure.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
